# 211Agent Playground

Use this notebook to manually test the agent. It can load either the full Indiana 211 dataset or the curated benchmark subset.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "agent211").exists() and (ROOT.parent / "agent211").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agent211 import Agent211, load_indiana_csv, load_resource_index
from agent211.llm import load_dotenv, make_openai_client, rerank_with_llm

load_dotenv(ROOT / ".env")

# Use the complete Indiana data for exploration. Switch to "curated" when testing benchmark behavior.
DATA_MODE = "full"  # "full" or "curated"

if DATA_MODE == "full":
    index = load_indiana_csv(ROOT / "data/indiana211/indiana211_resources_deduped.csv")
else:
    index = load_resource_index(ROOT / "data/indiana211/benchmark_curated/resource_index_curated.jsonl")

# For actual OpenAI tool-calling experiments, use PLANNER = "llm".
# Use PLANNER = "heuristic" only when you want a no-network baseline.
PLANNER = "llm"  # "llm" or "heuristic"
USE_RERANK = False
PROVIDER = "openrouter"  # "openrouter" or "openai"
MODEL = os.getenv("AGENT211_MODEL") or ("openai/gpt-4.1-mini" if PROVIDER == "openrouter" else "gpt-4.1-mini")

client = make_openai_client(PROVIDER) if PLANNER == "llm" or USE_RERANK else None
reranker = None
if USE_RERANK:
    reranker = lambda query, results, limit: rerank_with_llm(query, results, client, MODEL, limit=limit)

agent = Agent211(index, client=client, model=MODEL, use_openai_tools=(PLANNER == "llm"), reranker=reranker)

print(f"Loaded {len(index.resources)} resources from {DATA_MODE} data")
print(f"Planner: {PLANNER}; rerank: {USE_RERANK}; model: {MODEL}")
print(f"Categories: {index.benchmark_categories}")


In [ ]:
def ask(query: str, limit: int = 5):
    response = agent.ask(query, limit=limit)
    print("QUERY:", query)
    print("\nTOOL CALL:")
    print(response.tool_calls[0])
    print("\nANSWER:")
    print(response.answer)
    print("\nRAW RESULTS:")
    for result in response.results:
        r = result.resource
        print(
            f"{result.score:>5} | {r.resource_id} | {r.service_name} | "
            f"{r.agency_name} | {r.city} | {', '.join(r.service_area)}"
        )
        print("      matched_filters:", result.matched_filters)


## Try Queries


In [ ]:
ask("I need a food pantry in South Bend.")


In [ ]:
ask("My lights might be shut off and I need help with utilities in Lake County")


In [ ]:
ask("I need legal help with an eviction near Indianapolis")
